
# Financial Transaction Matching and Reconciliation Solution

This module provides a comprehensive framework for financial transaction matching
and account reconciliation in Snowflake, similar to BlackLine's capabilities.

The solution implements:
1. Financial transaction data generation (ERP, bank, credit card)
2. Transaction standardization and normalization
3. Rules-based matching algorithms
4. Machine learning scoring for fuzzy matches
5. Reconciliation workflows and exception handling
6. End-to-end orchestration with audit trails

Author: Jeremy Demlow 
Date: 2025-05-01


In [2]:
#| default_exp financial_matching

In [3]:
#| export
from typing import Dict, List, Optional, Tuple, Union
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, lit, regexp_replace, lower, upper, trim, lpad, round
from snowflake.snowpark.functions import abs as abs_func, datediff, to_date, concat
from snowflake.snowpark.functions import sum as sum_func, avg, count, to_char, current_timestamp
import json
import random
import string
from datetime import datetime, timedelta
import decimal
import uuid


# FINANCIAL DATA GENERATION
-----------------------------------------------------------------------------


In [4]:
#| export
def generate_financial_data(session: Session, 
                           num_transactions: int = 10000,
                           error_rate: float = 0.1,
                           date_range_start: str = "2023-01-01",
                           date_range_end: str = "2023-12-31",
                           erp_table: str = "ERP_TRANSACTIONS",
                           bank_table: str = "BANK_TRANSACTIONS",
                           credit_card_table: str = "CC_TRANSACTIONS") -> str:
    """
    Generates synthetic financial transaction data across multiple systems.
    
    Parameters:
        session (Session): Snowflake session
        num_transactions (int): Number of base transactions to generate
        error_rate (float): Probability of introducing errors/discrepancies (0-1)
        date_range_start (str): Start date for transactions (YYYY-MM-DD)
        date_range_end (str): End date for transactions (YYYY-MM-DD)
        erp_table (str): Name of ERP transactions table
        bank_table (str): Name of bank transactions table
        credit_card_table (str): Name of credit card transactions table
        
    Returns:
        str: Status message with data generation statistics
    """
    # Convert date strings to datetime objects
    start_date = datetime.strptime(date_range_start, "%Y-%m-%d")
    end_date = datetime.strptime(date_range_end, "%Y-%m-%d")
    date_range = (end_date - start_date).days
    
    # Define transaction types and their characteristics
    transaction_types = [
        {"type": "PURCHASE", "avg_amount": 1250.00, "std_dev": 750.00, "direction": -1},
        {"type": "SALE", "avg_amount": 1500.00, "std_dev": 1000.00, "direction": 1},
        {"type": "REFUND", "avg_amount": 350.00, "std_dev": 250.00, "direction": 1},
        {"type": "PAYMENT", "avg_amount": 1200.00, "std_dev": 800.00, "direction": -1},
        {"type": "FEE", "avg_amount": 50.00, "std_dev": 25.00, "direction": -1},
        {"type": "INTEREST", "avg_amount": 75.00, "std_dev": 40.00, "direction": 1},
        {"type": "TRANSFER", "avg_amount": 2500.00, "std_dev": 1500.00, "direction": 0}  # Can be either direction
    ]
    
    # Define vendors
    vendors = [
        "Acme Supplies", "TechCorp Solutions", "Global Logistics", "MetroShip Delivery",
        "Office Essentials", "DataSys Technologies", "IndustrialMart", "TravelPro Services",
        "Marketing Experts Inc.", "Financial Services Co.", "Legal Consulting Group",
        "CloudHost Solutions", "Utility Services", "Insurance Provider", "Maintenance Corp"
    ]
    
    # Define bank accounts and payment methods
    bank_accounts = [
        {"account_number": "1234567890", "account_name": "Operating Account", "bank": "First National Bank"},
        {"account_number": "2345678901", "account_name": "Payroll Account", "bank": "First National Bank"},
        {"account_number": "3456789012", "account_name": "Tax Account", "bank": "First National Bank"},
        {"account_number": "4567890123", "account_name": "Savings Account", "bank": "Community Bank"},
        {"account_number": "5678901234", "account_name": "Investment Account", "bank": "Investment Bank"}
    ]
    
    credit_cards = [
        {"card_number": "************1234", "card_type": "Visa", "issuer": "First National Bank"},
        {"card_number": "************5678", "card_type": "Mastercard", "issuer": "Community Bank"},
        {"card_number": "************9012", "card_type": "Amex", "issuer": "American Express"}
    ]
    
    payment_methods = [
        "ACH", "Wire", "Check", "Credit Card", "Cash", "Electronic Payment"
    ]
    
    # Define GL accounts
    gl_accounts = [
        {"code": "1010", "name": "Cash"},
        {"code": "1020", "name": "Accounts Receivable"},
        {"code": "1030", "name": "Inventory"},
        {"code": "1040", "name": "Prepaid Expenses"},
        {"code": "2010", "name": "Accounts Payable"},
        {"code": "2020", "name": "Accrued Expenses"},
        {"code": "2030", "name": "Taxes Payable"},
        {"code": "3010", "name": "Common Stock"},
        {"code": "4010", "name": "Sales Revenue"},
        {"code": "5010", "name": "Cost of Goods Sold"},
        {"code": "5020", "name": "Salaries Expense"},
        {"code": "5030", "name": "Marketing Expense"},
        {"code": "5040", "name": "Office Supplies"},
        {"code": "5050", "name": "Rent Expense"},
        {"code": "5060", "name": "Utilities Expense"}
    ]
    
    # Function to generate a random transaction date
    def random_date():
        return start_date + timedelta(days=random.randint(0, date_range))
    
    # Function to generate a random amount based on transaction type
    def random_amount(trans_type):
        type_info = next(t for t in transaction_types if t["type"] == trans_type)
        amount = random.normalvariate(type_info["avg_amount"], type_info["std_dev"])
        # Ensure amount is positive and has 2 decimal places
        # Avoid using round() by manually formatting to 2 decimal places
        amount = abs(amount)
        amount = int(amount * 100) / 100.0  # Truncate to 2 decimal places
        
        # Apply direction modifier
        if type_info["direction"] == -1 or (type_info["direction"] == 0 and random.random() < 0.5):
            amount = -amount
        return amount
    
    # Function to introduce errors into transaction data
    def introduce_error(value, error_type=None):
        if random.random() > error_rate:
            return value
            
        if error_type is None:
            error_type = random.choice(["typo", "truncate", "swap", "round", "date_shift", "amount_change"])
            
        if isinstance(value, str):
            if error_type == "typo":
                # Introduce a typo
                if len(value) > 1:
                    pos = random.randint(0, len(value) - 1)
                    chars = list(value)
                    chars[pos] = random.choice(string.ascii_letters + string.digits)
                    return ''.join(chars)
            elif error_type == "truncate":
                # Truncate the value
                if len(value) > 3:
                    trunc_len = random.randint(1, len(value) // 3)
                    return value[:-trunc_len]
            elif error_type == "swap":
                # Swap adjacent characters
                if len(value) > 2:
                    pos = random.randint(0, len(value) - 2)
                    chars = list(value)
                    chars[pos], chars[pos+1] = chars[pos+1], chars[pos]
                    return ''.join(chars)
        elif isinstance(value, datetime):
            if error_type == "date_shift":
                # Shift date by a few days
                shift_days = random.randint(-5, 5)
                return value + timedelta(days=shift_days)
        elif isinstance(value, (int, float, decimal.Decimal)):
            if error_type == "round":
                # Round to nearest 10 without using round()
                return int(value / 10) * 10.0
            elif error_type == "amount_change":
                # Change amount slightly
                modifier = random.uniform(0.95, 1.05)
                # Format to 2 decimal places without using round()
                result = value * modifier
                return int(result * 100) / 100.0
                
        return value
    
    # Generate base transactions
    base_transactions = []
    for i in range(num_transactions):
        trans_id = str(uuid.uuid4())
        trans_type = random.choice(transaction_types)["type"]
        trans_date = random_date()
        vendor = random.choice(vendors)
        amount = random_amount(trans_type)
        bank_account = random.choice(bank_accounts)
        gl_account_debit = random.choice(gl_accounts)
        gl_account_credit = random.choice(gl_accounts)
        payment_method = random.choice(payment_methods)
        reference = f"REF-{random.randint(100000, 999999)}"
        description = f"{trans_type} - {vendor}"
        
        base_transactions.append({
            "transaction_id": trans_id,
            "transaction_type": trans_type,
            "transaction_date": trans_date,
            "post_date": trans_date + timedelta(days=random.randint(0, 3)),
            "vendor": vendor,
            "amount": amount,
            "currency": "USD",
            "bank_account": bank_account,
            "gl_account_debit": gl_account_debit,
            "gl_account_credit": gl_account_credit,
            "payment_method": payment_method,
            "reference": reference,
            "description": description
        })
    
    # Generate ERP transactions (all base transactions present)
    erp_data = []
    for trans in base_transactions:
        erp_record = [
            f"ERP-{trans['transaction_id'][:8]}",
            trans['transaction_type'],
            trans['transaction_date'].strftime("%Y-%m-%d"),
            trans['post_date'].strftime("%Y-%m-%d"),
            trans['vendor'],
            trans['amount'],
            trans['currency'],
            trans['bank_account']['account_number'],
            trans['bank_account']['account_name'],
            trans['gl_account_debit']['code'],
            trans['gl_account_credit']['code'],
            trans['payment_method'],
            trans['reference'],
            trans['description'],
            f"USER-{random.randint(1000, 9999)}",
            datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        ]
        erp_data.append(erp_record)
    
    # Generate bank transactions (only those with bank payment methods, some with errors)
    bank_data = []
    bank_payment_methods = ["ACH", "Wire", "Check", "Electronic Payment"]
    for trans in base_transactions:
        if trans['payment_method'] in bank_payment_methods:
            # Skip some transactions (simulating missing transactions in bank data)
            if random.random() < 0.05:
                continue
                
            # Generate bank transaction record
            bank_trans_date = introduce_error(trans['transaction_date'], "date_shift")
            bank_amount = introduce_error(trans['amount'], "amount_change")
            bank_description = introduce_error(trans['description'], "typo")
            
            bank_record = [
                f"BANK-{trans['transaction_id'][:8]}",
                trans['transaction_type'],
                bank_trans_date.strftime("%Y-%m-%d"),
                trans['bank_account']['account_number'],
                trans['bank_account']['bank'],
                bank_amount,
                trans['currency'],
                trans['payment_method'],
                trans['reference'],
                bank_description,
                "PROCESSED",
                datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            ]
            bank_data.append(bank_record)
            
    # Generate credit card transactions
    cc_data = []
    for trans in base_transactions:
        if trans['payment_method'] == "Credit Card":
            # Skip some transactions (simulating pending or missing cc transactions)
            if random.random() < 0.08:
                continue
                
            # Pick a credit card
            credit_card = random.choice(credit_cards)
            
            # Generate credit card transaction record
            cc_trans_date = introduce_error(trans['transaction_date'], "date_shift")
            cc_post_date = cc_trans_date + timedelta(days=random.randint(1, 5))
            cc_amount = introduce_error(trans['amount'], "amount_change")
            cc_vendor = introduce_error(trans['vendor'], "typo")
            
            cc_record = [
                f"CC-{trans['transaction_id'][:8]}",
                credit_card['card_number'],
                credit_card['card_type'],
                credit_card['issuer'],
                cc_trans_date.strftime("%Y-%m-%d"),
                cc_post_date.strftime("%Y-%m-%d"),
                cc_amount,
                trans['currency'],
                cc_vendor,
                trans['reference'],
                "POSTED",
                datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            ]
            cc_data.append(cc_record)
    
    # Create Snowflake DataFrames
    erp_df = session.create_dataframe(
        erp_data,
        schema=["TRANSACTION_ID", "TRANSACTION_TYPE", "TRANSACTION_DATE", "POST_DATE", 
                "VENDOR", "AMOUNT", "CURRENCY", "ACCOUNT_NUMBER", "ACCOUNT_NAME", 
                "GL_ACCOUNT_DEBIT", "GL_ACCOUNT_CREDIT", "PAYMENT_METHOD", 
                "REFERENCE", "DESCRIPTION", "CREATED_BY", "CREATED_AT"]
    )
    
    bank_df = session.create_dataframe(
        bank_data,
        schema=["TRANSACTION_ID", "TRANSACTION_TYPE", "TRANSACTION_DATE", 
                "ACCOUNT_NUMBER", "BANK_NAME", "AMOUNT", "CURRENCY", 
                "PAYMENT_METHOD", "REFERENCE", "DESCRIPTION", 
                "STATUS", "PROCESSED_AT"]
    )
    
    cc_df = session.create_dataframe(
        cc_data,
        schema=["TRANSACTION_ID", "CARD_NUMBER", "CARD_TYPE", "CARD_ISSUER",
                "TRANSACTION_DATE", "POST_DATE", "AMOUNT", "CURRENCY", 
                "MERCHANT", "REFERENCE", "STATUS", "PROCESSED_AT"]
    )
    
    # Save to tables
    erp_df.write.mode("overwrite").save_as_table(erp_table)
    bank_df.write.mode("overwrite").save_as_table(bank_table)
    cc_df.write.mode("overwrite").save_as_table(credit_card_table)
    
    # Return summary
    stats = {
        "total_base_transactions": len(base_transactions),
        "erp_transactions": len(erp_data),
        "bank_transactions": len(bank_data),
        "credit_card_transactions": len(cc_data),
        "error_rate_applied": error_rate,
        "date_range": f"{date_range_start} to {date_range_end}",
        "tables_created": [erp_table, bank_table, credit_card_table]
    }
    
    return json.dumps(stats)



# FINANCIAL DATA STANDARDIZATION
-----------------------------------------------------------------------------


In [5]:
#| export
def standardize_erp_transactions(session: Session, input_table: str, output_table: str) -> str:
    """
    Standardizes ERP transaction data for matching.
    
    Parameters:
        session (Session): Snowflake session
        input_table (str): Source ERP transactions table
        output_table (str): Target standardized table
        
    Returns:
        str: Status message
    """
    # Load the input table
    df = session.table(input_table)
    
    # Standardize transaction date to DATE type
    df = df.with_column("TRANSACTION_DATE_STD", to_date(col("TRANSACTION_DATE")))
    
    # Standardize amount (ensure 2 decimal places)
    df = df.with_column("AMOUNT_STD", round(col("AMOUNT"), 2))
    
    # Standardize account number (remove non-alphanumeric characters)
    df = df.with_column("ACCOUNT_NUMBER_STD", regexp_replace(col("ACCOUNT_NUMBER"), "[^A-Z0-9]", ""))
    
    # Standardize reference number (uppercase, remove spaces)
    df = df.with_column("REFERENCE_STD", upper(regexp_replace(col("REFERENCE"), "\\s+", "")))
    
    # Create matching keys
    # Key 1: Date + Amount + Account (strict match)
    df = df.with_column(
        "MATCH_KEY_1",
        concat(
            to_char(col("TRANSACTION_DATE_STD"), "YYYYMMDD"),
            lit("_"),
            to_char(abs_func(col("AMOUNT_STD")), "0000000.00"),
            lit("_"),
            col("ACCOUNT_NUMBER_STD")
        )
    )
    
    # Key 2: Date ± 3 days + Amount + Account (date flexibility)
    # This is a template key - actual match keys will be generated in the matching process
    
    # Key 3: Reference number
    df = df.with_column("MATCH_KEY_3", col("REFERENCE_STD"))
    
    # Save to output table
    df.write.mode("overwrite").save_as_table(output_table)
    
    return f"Standardized ERP transactions from {input_table} and saved to {output_table}"


In [6]:
#| export
def standardize_bank_transactions(session: Session, input_table: str, output_table: str) -> str:
    """
    Standardizes bank transaction data for matching.
    
    Parameters:
        session (Session): Snowflake session
        input_table (str): Source bank transactions table
        output_table (str): Target standardized table
        
    Returns:
        str: Status message
    """
    # Load the input table
    df = session.table(input_table)
    
    # Standardize transaction date to DATE type
    df = df.with_column("TRANSACTION_DATE_STD", to_date(col("TRANSACTION_DATE")))
    
    # Standardize amount (ensure 2 decimal places)
    df = df.with_column("AMOUNT_STD", round(col("AMOUNT"), 2))
    
    # Standardize account number (remove non-alphanumeric characters)
    df = df.with_column("ACCOUNT_NUMBER_STD", regexp_replace(col("ACCOUNT_NUMBER"), "[^A-Z0-9]", ""))
    
    # Standardize reference number (uppercase, remove spaces)
    df = df.with_column("REFERENCE_STD", upper(regexp_replace(col("REFERENCE"), "\\s+", "")))
    
    # Create matching keys (same as ERP for consistency)
    # Key 1: Date + Amount + Account (strict match)
    df = df.with_column(
        "MATCH_KEY_1",
        concat(
            to_char(col("TRANSACTION_DATE_STD"), "YYYYMMDD"),
            lit("_"),
            to_char(abs_func(col("AMOUNT_STD")), "0000000.00"),
            lit("_"),
            col("ACCOUNT_NUMBER_STD")
        )
    )
    
    # Key 3: Reference number
    df = df.with_column("MATCH_KEY_3", col("REFERENCE_STD"))
    
    # Save to output table
    df.write.mode("overwrite").save_as_table(output_table)
    
    return f"Standardized bank transactions from {input_table} and saved to {output_table}"


In [7]:
#| export
def standardize_cc_transactions(session: Session, input_table: str, output_table: str) -> str:
    """
    Standardizes credit card transaction data for matching.
    
    Parameters:
        session (Session): Snowflake session
        input_table (str): Source credit card transactions table
        output_table (str): Target standardized table
        
    Returns:
        str: Status message
    """
    # Load the input table
    df = session.table(input_table)
    
    # Standardize transaction date to DATE type
    df = df.with_column("TRANSACTION_DATE_STD", to_date(col("TRANSACTION_DATE")))
    
    # Standardize post date to DATE type
    df = df.with_column("POST_DATE_STD", to_date(col("POST_DATE")))
    
    # Standardize amount (ensure 2 decimal places)
    df = df.with_column("AMOUNT_STD", round(col("AMOUNT"), 2))
    
    # Standardize card number (last 4 digits only for matching)
    df = df.with_column("CARD_LAST4", regexp_replace(col("CARD_NUMBER"), ".*([0-9]{4})", "$1"))
    
    # Standardize merchant name (uppercase, remove common words)
    df = df.with_column(
        "MERCHANT_STD", 
        upper(
            regexp_replace(
                regexp_replace(col("MERCHANT"), "\\s+(INC|LLC|CORP|CO|LTD)\\.?\\s*$", ""),
                "\\s+", " "
            )
        )
    )
    
    # Standardize reference number (uppercase, remove spaces)
    df = df.with_column("REFERENCE_STD", upper(regexp_replace(col("REFERENCE"), "\\s+", "")))
    
    # Create matching keys
    # Key 1: Date + Amount + Last4 (specific to CC transactions)
    df = df.with_column(
        "MATCH_KEY_1",
        concat(
            to_char(col("TRANSACTION_DATE_STD"), "YYYYMMDD"),
            lit("_"),
            to_char(abs_func(col("AMOUNT_STD")), "0000000.00"),
            lit("_"),
            col("CARD_LAST4")
        )
    )
    
    # Key 2: Post Date + Amount + Last4 (for matching with post date)
    df = df.with_column(
        "MATCH_KEY_2",
        concat(
            to_char(col("POST_DATE_STD"), "YYYYMMDD"),
            lit("_"),
            to_char(abs_func(col("AMOUNT_STD")), "0000000.00"),
            lit("_"),
            col("CARD_LAST4")
        )
    )
    
    # Key 3: Reference number
    df = df.with_column("MATCH_KEY_3", col("REFERENCE_STD"))
    
    # Save to output table
    df.write.mode("overwrite").save_as_table(output_table)
    
    return f"Standardized credit card transactions from {input_table} and saved to {output_table}"




# TRANSACTION MATCHING
-----------------------------------------------------------------------------


In [8]:
#| export
def match_erp_to_bank(session: Session,
                     erp_table: str,
                     bank_table: str,
                     output_table: str,
                     date_variance: int = 3) -> str:
    """
    Matches ERP transactions to bank transactions.
    
    Parameters:
        session (Session): Snowflake session
        erp_table (str): Standardized ERP transactions table
        bank_table (str): Standardized bank transactions table
        output_table (str): Output table for match results
        date_variance (int): Number of days to allow for date differences
        
    Returns:
        str: Status message with match statistics
    """
    # Define rules for matching in order of confidence
    
    # Rule 1: Exact match on date + amount + account
    rule1_sql = f"""
    SELECT 
        e.TRANSACTION_ID as ERP_TRANSACTION_ID,
        b.TRANSACTION_ID as BANK_TRANSACTION_ID,
        e.TRANSACTION_DATE_STD as ERP_DATE,
        b.TRANSACTION_DATE_STD as BANK_DATE,
        e.AMOUNT_STD as ERP_AMOUNT,
        b.AMOUNT_STD as BANK_AMOUNT,
        e.ACCOUNT_NUMBER_STD as ERP_ACCOUNT,
        b.ACCOUNT_NUMBER_STD as BANK_ACCOUNT,
        e.REFERENCE_STD as ERP_REFERENCE,
        b.REFERENCE_STD as BANK_REFERENCE,
        e.DESCRIPTION as ERP_DESCRIPTION,
        b.DESCRIPTION as BANK_DESCRIPTION,
        1 as MATCH_RULE,
        'EXACT_MATCH' as MATCH_TYPE,
        1.0 as MATCH_CONFIDENCE,
        0 as DATE_DIFFERENCE,
        0.0 as AMOUNT_DIFFERENCE,
        CURRENT_TIMESTAMP() as MATCHED_AT
    FROM {erp_table} e
    JOIN {bank_table} b ON e.MATCH_KEY_1 = b.MATCH_KEY_1
    """
    
    # Rule 2: Reference number match
    rule2_sql = f"""
    SELECT 
        e.TRANSACTION_ID as ERP_TRANSACTION_ID,
        b.TRANSACTION_ID as BANK_TRANSACTION_ID,
        e.TRANSACTION_DATE_STD as ERP_DATE,
        b.TRANSACTION_DATE_STD as BANK_DATE,
        e.AMOUNT_STD as ERP_AMOUNT,
        b.AMOUNT_STD as BANK_AMOUNT,
        e.ACCOUNT_NUMBER_STD as ERP_ACCOUNT,
        b.ACCOUNT_NUMBER_STD as BANK_ACCOUNT,
        e.REFERENCE_STD as ERP_REFERENCE,
        b.REFERENCE_STD as BANK_REFERENCE,
        e.DESCRIPTION as ERP_DESCRIPTION,
        b.DESCRIPTION as BANK_DESCRIPTION,
        2 as MATCH_RULE,
        'REFERENCE_MATCH' as MATCH_TYPE,
        0.95 as MATCH_CONFIDENCE,
        DATEDIFF('DAY', e.TRANSACTION_DATE_STD, b.TRANSACTION_DATE_STD) as DATE_DIFFERENCE,
        ABS(e.AMOUNT_STD - b.AMOUNT_STD) as AMOUNT_DIFFERENCE,
        CURRENT_TIMESTAMP() as MATCHED_AT
    FROM {erp_table} e
    JOIN {bank_table} b ON e.REFERENCE_STD = b.REFERENCE_STD
        AND e.REFERENCE_STD IS NOT NULL AND b.REFERENCE_STD IS NOT NULL
        AND e.REFERENCE_STD != ''
        AND e.ACCOUNT_NUMBER_STD = b.ACCOUNT_NUMBER_STD
        AND ABS(e.AMOUNT_STD - b.AMOUNT_STD) < 0.01  -- amounts must be very close
    WHERE NOT EXISTS (
        SELECT 1 FROM {erp_table} e2
        JOIN {bank_table} b2 ON e2.MATCH_KEY_1 = b2.MATCH_KEY_1
        WHERE e2.TRANSACTION_ID = e.TRANSACTION_ID
        OR b2.TRANSACTION_ID = b.TRANSACTION_ID
    )
    """
    
    # Rule 3: Date within variance + amount + account
    # This uses a dynamic approach to handle date variance
    date_variance_matches = []
    for day_diff in range(1, date_variance + 1):
        for direction in ["+", "-"]:
            date_variance_sql = f"""
            SELECT 
                e.TRANSACTION_ID as ERP_TRANSACTION_ID,
                b.TRANSACTION_ID as BANK_TRANSACTION_ID,
                e.TRANSACTION_DATE_STD as ERP_DATE,
                b.TRANSACTION_DATE_STD as BANK_DATE,
                e.AMOUNT_STD as ERP_AMOUNT,
                b.AMOUNT_STD as BANK_AMOUNT,
                e.ACCOUNT_NUMBER_STD as ERP_ACCOUNT,
                b.ACCOUNT_NUMBER_STD as BANK_ACCOUNT,
                e.REFERENCE_STD as ERP_REFERENCE,
                b.REFERENCE_STD as BANK_REFERENCE,
                e.DESCRIPTION as ERP_DESCRIPTION,
                b.DESCRIPTION as BANK_DESCRIPTION,
                3 as MATCH_RULE,
                'DATE_VARIANCE_MATCH' as MATCH_TYPE,
                (1.0 - ({day_diff} * 0.1)) as MATCH_CONFIDENCE,
                {day_diff} * {1 if direction == "+" else -1} as DATE_DIFFERENCE,
                ABS(e.AMOUNT_STD - b.AMOUNT_STD) as AMOUNT_DIFFERENCE,
                CURRENT_TIMESTAMP() as MATCHED_AT
            FROM {erp_table} e
            JOIN {bank_table} b ON 
                DATEADD('DAY', {direction}{day_diff}, e.TRANSACTION_DATE_STD) = b.TRANSACTION_DATE_STD
                AND ABS(e.AMOUNT_STD) = ABS(b.AMOUNT_STD)
                AND e.ACCOUNT_NUMBER_STD = b.ACCOUNT_NUMBER_STD
            WHERE NOT EXISTS (
                SELECT 1 FROM {erp_table} e2
                JOIN {bank_table} b2 ON e2.MATCH_KEY_1 = b2.MATCH_KEY_1
                WHERE e2.TRANSACTION_ID = e.TRANSACTION_ID
                OR b2.TRANSACTION_ID = b.TRANSACTION_ID
            )
            AND NOT EXISTS (
                SELECT 1 FROM {erp_table} e2
                JOIN {bank_table} b2 ON e2.REFERENCE_STD = b2.REFERENCE_STD
                    AND e2.REFERENCE_STD IS NOT NULL AND b2.REFERENCE_STD IS NOT NULL
                WHERE e2.TRANSACTION_ID = e.TRANSACTION_ID
                OR b2.TRANSACTION_ID = b.TRANSACTION_ID
            )
            """
            date_variance_matches.append(date_variance_sql)
    
    # Rule 4: Amount-based match with small variance (e.g., rounding differences)
    rule4_sql = f"""
    SELECT 
        e.TRANSACTION_ID as ERP_TRANSACTION_ID,
        b.TRANSACTION_ID as BANK_TRANSACTION_ID,
        e.TRANSACTION_DATE_STD as ERP_DATE,
        b.TRANSACTION_DATE_STD as BANK_DATE,
        e.AMOUNT_STD as ERP_AMOUNT,
        b.AMOUNT_STD as BANK_AMOUNT,
        e.ACCOUNT_NUMBER_STD as ERP_ACCOUNT,
        b.ACCOUNT_NUMBER_STD as BANK_ACCOUNT,
        e.REFERENCE_STD as ERP_REFERENCE,
        b.REFERENCE_STD as BANK_REFERENCE,
        e.DESCRIPTION as ERP_DESCRIPTION,
        b.DESCRIPTION as BANK_DESCRIPTION,
        4 as MATCH_RULE,
        'AMOUNT_VARIANCE_MATCH' as MATCH_TYPE,
        0.7 - (ABS(e.AMOUNT_STD - b.AMOUNT_STD) / GREATEST(ABS(e.AMOUNT_STD), ABS(b.AMOUNT_STD))) as MATCH_CONFIDENCE,
        DATEDIFF('DAY', e.TRANSACTION_DATE_STD, b.TRANSACTION_DATE_STD) as DATE_DIFFERENCE,
        ABS(e.AMOUNT_STD - b.AMOUNT_STD) as AMOUNT_DIFFERENCE,
        CURRENT_TIMESTAMP() as MATCHED_AT
    FROM {erp_table} e
    JOIN {bank_table} b ON 
        ABS(e.TRANSACTION_DATE_STD - b.TRANSACTION_DATE_STD) <= {date_variance}
        AND ABS(e.AMOUNT_STD - b.AMOUNT_STD) / GREATEST(ABS(e.AMOUNT_STD), ABS(b.AMOUNT_STD)) < 0.05  -- 5% tolerance
        AND e.ACCOUNT_NUMBER_STD = b.ACCOUNT_NUMBER_STD
    WHERE NOT EXISTS (
        SELECT 1 FROM {erp_table} e2
        JOIN {bank_table} b2 ON e2.MATCH_KEY_1 = b2.MATCH_KEY_1
        WHERE e2.TRANSACTION_ID = e.TRANSACTION_ID
        OR b2.TRANSACTION_ID = b.TRANSACTION_ID
    )
    AND NOT EXISTS (
        SELECT 1 FROM {erp_table} e2
        JOIN {bank_table} b2 ON e2.REFERENCE_STD = b2.REFERENCE_STD
            AND e2.REFERENCE_STD IS NOT NULL AND b2.REFERENCE_STD IS NOT NULL
        WHERE e2.TRANSACTION_ID = e.TRANSACTION_ID
        OR b2.TRANSACTION_ID = b.TRANSACTION_ID
    )
    """
    
    # Combine all matching rules with UNION ALL, ordered by rule priority
    matches_sql = f"""
    {rule1_sql}
    UNION ALL
    {rule2_sql}
    UNION ALL
    {' UNION ALL '.join(date_variance_matches)}
    UNION ALL
    {rule4_sql}
    """
    
    # Handle potential duplicates by selecting highest confidence matches
    final_sql = f"""
    WITH all_matches AS (
        {matches_sql}
    ),
    ranked_matches AS (
        SELECT 
            *,
            ROW_NUMBER() OVER (PARTITION BY ERP_TRANSACTION_ID ORDER BY MATCH_RULE, MATCH_CONFIDENCE DESC) as erp_rank,
            ROW_NUMBER() OVER (PARTITION BY BANK_TRANSACTION_ID ORDER BY MATCH_RULE, MATCH_CONFIDENCE DESC) as bank_rank
        FROM all_matches
    )
    SELECT 
        *,
        CASE 
            WHEN MATCH_CONFIDENCE >= 0.95 THEN 'Auto-Match'
            WHEN MATCH_CONFIDENCE >= 0.8 THEN 'Suggested-Match'
            ELSE 'Potential-Match'
        END as MATCH_STATUS
    FROM ranked_matches
    WHERE erp_rank = 1 AND bank_rank = 1
    """
    
    # Execute and save results
    result_df = session.sql(final_sql)
    result_df.write.mode("overwrite").save_as_table(output_table)
    
    # Get match statistics
    stats_sql = f"""
    SELECT
        MATCH_STATUS,
        COUNT(*) as MATCH_COUNT,
        AVG(MATCH_CONFIDENCE) as AVG_CONFIDENCE,
        MIN(MATCH_CONFIDENCE) as MIN_CONFIDENCE,
        MAX(MATCH_CONFIDENCE) as MAX_CONFIDENCE
    FROM {output_table}
    GROUP BY MATCH_STATUS
    ORDER BY MATCH_STATUS
    """
    
    stats_df = session.sql(stats_sql)
    stats_rows = stats_df.collect()
    
    # Fix: Convert Row objects to dictionaries properly
    stats_dicts = []
    for row in stats_rows:
        # Create a dictionary with each column name and value
        row_dict = {}
        for col_name in stats_df.columns:
            row_dict[col_name] = row[col_name]
        stats_dicts.append(row_dict)
    
    # Count unmatched transactions
    unmatched_sql = f"""
    SELECT
        'Unmatched ERP' as CATEGORY,
        COUNT(*) as COUNT
    FROM {erp_table} e
    WHERE NOT EXISTS (
        SELECT 1 FROM {output_table} m
        WHERE m.ERP_TRANSACTION_ID = e.TRANSACTION_ID
    )
    
    UNION ALL
    
    SELECT
        'Unmatched Bank' as CATEGORY,
        COUNT(*) as COUNT
    FROM {bank_table} b
    WHERE NOT EXISTS (
        SELECT 1 FROM {output_table} m
        WHERE m.BANK_TRANSACTION_ID = b.TRANSACTION_ID
    )
    """
    
    unmatched_df = session.sql(unmatched_sql)
    unmatched_rows = unmatched_df.collect()
    
    # Fix: Convert Row objects to dictionaries properly
    unmatched_dicts = []
    for row in unmatched_rows:
        # Create a dictionary with each column name and value
        row_dict = {}
        for col_name in unmatched_df.columns:
            row_dict[col_name] = row[col_name]
        unmatched_dicts.append(row_dict)
    
    # Return summary
    match_results = {
        "match_statistics": stats_dicts,
        "unmatched_statistics": unmatched_dicts,
        "source_tables": [erp_table, bank_table],
        "target_table": output_table
    }
    
    return json.dumps(match_results)


In [9]:
#| export

def match_erp_to_credit_card(session: Session,
                            erp_table: str,
                            cc_table: str,
                            output_table: str,
                            date_variance: int = 5) -> str:
    """
    Matches ERP credit card transactions to credit card statement data.
    
    Parameters:
        session (Session): Snowflake session
        erp_table (str): Standardized ERP transactions table
        cc_table (str): Standardized credit card transactions table
        output_table (str): Output table for match results
        date_variance (int): Number of days to allow for date differences
        
    Returns:
        str: Status message with match statistics
    """
    # Define rules for matching in order of confidence
    
    # Rule 1: Exact match on date + amount + reference
    rule1_sql = f"""
    SELECT 
        e.TRANSACTION_ID as ERP_TRANSACTION_ID,
        c.TRANSACTION_ID as CC_TRANSACTION_ID,
        e.TRANSACTION_DATE_STD as ERP_DATE,
        c.TRANSACTION_DATE_STD as CC_DATE,
        e.AMOUNT_STD as ERP_AMOUNT,
        c.AMOUNT_STD as CC_AMOUNT,
        e.PAYMENT_METHOD as ERP_PAYMENT_METHOD,
        c.CARD_NUMBER as CC_CARD_NUMBER,
        e.REFERENCE_STD as ERP_REFERENCE,
        c.REFERENCE_STD as CC_REFERENCE,
        e.VENDOR as ERP_VENDOR,
        c.MERCHANT_STD as CC_MERCHANT,
        1 as MATCH_RULE,
        'EXACT_MATCH' as MATCH_TYPE,
        1.0 as MATCH_CONFIDENCE,
        0 as DATE_DIFFERENCE,
        0.0 as AMOUNT_DIFFERENCE,
        CURRENT_TIMESTAMP() as MATCHED_AT
    FROM {erp_table} e
    JOIN {cc_table} c ON 
        e.TRANSACTION_DATE_STD = c.TRANSACTION_DATE_STD
        AND ABS(e.AMOUNT_STD) = ABS(c.AMOUNT_STD)
        AND e.REFERENCE_STD = c.REFERENCE_STD
        AND e.PAYMENT_METHOD = 'Credit Card'
    """
    
    # Rule 2: Post date + amount match
    rule2_sql = f"""
    SELECT 
        e.TRANSACTION_ID as ERP_TRANSACTION_ID,
        c.TRANSACTION_ID as CC_TRANSACTION_ID,
        e.TRANSACTION_DATE_STD as ERP_DATE,
        c.POST_DATE_STD as CC_DATE,
        e.AMOUNT_STD as ERP_AMOUNT,
        c.AMOUNT_STD as CC_AMOUNT,
        e.PAYMENT_METHOD as ERP_PAYMENT_METHOD,
        c.CARD_NUMBER as CC_CARD_NUMBER,
        e.REFERENCE_STD as ERP_REFERENCE,
        c.REFERENCE_STD as CC_REFERENCE,
        e.VENDOR as ERP_VENDOR,
        c.MERCHANT_STD as CC_MERCHANT,
        2 as MATCH_RULE,
        'POST_DATE_MATCH' as MATCH_TYPE,
        0.95 as MATCH_CONFIDENCE,
        DATEDIFF('DAY', e.TRANSACTION_DATE_STD, c.POST_DATE_STD) as DATE_DIFFERENCE,
        ABS(e.AMOUNT_STD - c.AMOUNT_STD) as AMOUNT_DIFFERENCE,
        CURRENT_TIMESTAMP() as MATCHED_AT
    FROM {erp_table} e
    JOIN {cc_table} c ON 
        e.TRANSACTION_DATE_STD = c.POST_DATE_STD
        AND ABS(e.AMOUNT_STD) = ABS(c.AMOUNT_STD)
        AND e.PAYMENT_METHOD = 'Credit Card'
    WHERE NOT EXISTS (
        SELECT 1 FROM {erp_table} e2
        JOIN {cc_table} c2 ON 
            e2.TRANSACTION_DATE_STD = c2.TRANSACTION_DATE_STD
            AND ABS(e2.AMOUNT_STD) = ABS(c2.AMOUNT_STD)
            AND e2.REFERENCE_STD = c2.REFERENCE_STD
        WHERE e2.TRANSACTION_ID = e.TRANSACTION_ID
        OR c2.TRANSACTION_ID = c.TRANSACTION_ID
    )
    """
    
    # Rule 3: Merchant/vendor + date within variance + amount
    rule3_sql = f"""
    SELECT 
        e.TRANSACTION_ID as ERP_TRANSACTION_ID,
        c.TRANSACTION_ID as CC_TRANSACTION_ID,
        e.TRANSACTION_DATE_STD as ERP_DATE,
        c.TRANSACTION_DATE_STD as CC_DATE,
        e.AMOUNT_STD as ERP_AMOUNT,
        c.AMOUNT_STD as CC_AMOUNT,
        e.PAYMENT_METHOD as ERP_PAYMENT_METHOD,
        c.CARD_NUMBER as CC_CARD_NUMBER,
        e.REFERENCE_STD as ERP_REFERENCE,
        c.REFERENCE_STD as CC_REFERENCE,
        e.VENDOR as ERP_VENDOR,
        c.MERCHANT_STD as CC_MERCHANT,
        3 as MATCH_RULE,
        'MERCHANT_DATE_AMOUNT_MATCH' as MATCH_TYPE,
        0.9 as MATCH_CONFIDENCE,
        DATEDIFF('DAY', e.TRANSACTION_DATE_STD, c.TRANSACTION_DATE_STD) as DATE_DIFFERENCE,
        ABS(e.AMOUNT_STD - c.AMOUNT_STD) as AMOUNT_DIFFERENCE,
        CURRENT_TIMESTAMP() as MATCHED_AT
    FROM {erp_table} e
    JOIN {cc_table} c ON 
        ABS(DATEDIFF('DAY', e.TRANSACTION_DATE_STD, c.TRANSACTION_DATE_STD)) <= {date_variance}
        AND ABS(e.AMOUNT_STD) = ABS(c.AMOUNT_STD)
        AND UPPER(e.VENDOR) = c.MERCHANT_STD
        AND e.PAYMENT_METHOD = 'Credit Card'
    WHERE NOT EXISTS (
        SELECT 1 FROM {erp_table} e2
        JOIN {cc_table} c2 ON 
            e2.TRANSACTION_DATE_STD = c2.TRANSACTION_DATE_STD
            AND ABS(e2.AMOUNT_STD) = ABS(c2.AMOUNT_STD)
        WHERE e2.TRANSACTION_ID = e.TRANSACTION_ID
        OR c2.TRANSACTION_ID = c.TRANSACTION_ID
    )
    AND NOT EXISTS (
        SELECT 1 FROM {erp_table} e2
        JOIN {cc_table} c2 ON 
            e2.TRANSACTION_DATE_STD = c2.POST_DATE_STD
            AND ABS(e2.AMOUNT_STD) = ABS(c2.AMOUNT_STD)
        WHERE e2.TRANSACTION_ID = e.TRANSACTION_ID
        OR c2.TRANSACTION_ID = c.TRANSACTION_ID
    )
    """
    
    # Rule 4: Fuzzy match with date and amount variance
    rule4_sql = f"""
    SELECT 
        e.TRANSACTION_ID as ERP_TRANSACTION_ID,
        c.TRANSACTION_ID as CC_TRANSACTION_ID,
        e.TRANSACTION_DATE_STD as ERP_DATE,
        c.TRANSACTION_DATE_STD as CC_DATE,
        e.AMOUNT_STD as ERP_AMOUNT,
        c.AMOUNT_STD as CC_AMOUNT,
        e.PAYMENT_METHOD as ERP_PAYMENT_METHOD,
        c.CARD_NUMBER as CC_CARD_NUMBER,
        e.REFERENCE_STD as ERP_REFERENCE,
        c.REFERENCE_STD as CC_REFERENCE,
        e.VENDOR as ERP_VENDOR,
        c.MERCHANT_STD as CC_MERCHANT,
        4 as MATCH_RULE,
        'FUZZY_MATCH' as MATCH_TYPE,
        0.7 - (ABS(e.AMOUNT_STD - c.AMOUNT_STD) / GREATEST(ABS(e.AMOUNT_STD), ABS(c.AMOUNT_STD))) 
             - (ABS(DATEDIFF('DAY', e.TRANSACTION_DATE_STD, c.TRANSACTION_DATE_STD)) * 0.02) as MATCH_CONFIDENCE,
        DATEDIFF('DAY', e.TRANSACTION_DATE_STD, c.TRANSACTION_DATE_STD) as DATE_DIFFERENCE,
        ABS(e.AMOUNT_STD - c.AMOUNT_STD) as AMOUNT_DIFFERENCE,
        CURRENT_TIMESTAMP() as MATCHED_AT
    FROM {erp_table} e
    JOIN {cc_table} c ON 
        ABS(DATEDIFF('DAY', e.TRANSACTION_DATE_STD, c.TRANSACTION_DATE_STD)) <= {date_variance}
        AND ABS(e.AMOUNT_STD - c.AMOUNT_STD) / GREATEST(ABS(e.AMOUNT_STD), ABS(c.AMOUNT_STD)) < 0.05  -- 5% tolerance
        AND e.PAYMENT_METHOD = 'Credit Card'
    WHERE NOT EXISTS (
        SELECT 1 FROM {erp_table} e2
        JOIN {cc_table} c2 ON 
            (e2.TRANSACTION_DATE_STD = c2.TRANSACTION_DATE_STD OR e2.TRANSACTION_DATE_STD = c2.POST_DATE_STD)
            AND ABS(e2.AMOUNT_STD) = ABS(c2.AMOUNT_STD)
        WHERE e2.TRANSACTION_ID = e.TRANSACTION_ID
        OR c2.TRANSACTION_ID = c.TRANSACTION_ID
    )
    AND NOT EXISTS (
        SELECT 1 FROM {erp_table} e2
        JOIN {cc_table} c2 ON 
            ABS(DATEDIFF('DAY', e2.TRANSACTION_DATE_STD, c2.TRANSACTION_DATE_STD)) <= {date_variance}
            AND ABS(e2.AMOUNT_STD) = ABS(c2.AMOUNT_STD)
            AND UPPER(e2.VENDOR) = c2.MERCHANT_STD
        WHERE e2.TRANSACTION_ID = e.TRANSACTION_ID
        OR c2.TRANSACTION_ID = c.TRANSACTION_ID
    )
    """
    
    # Combine all matching rules with UNION ALL, ordered by rule priority
    matches_sql = f"""
    {rule1_sql}
    UNION ALL
    {rule2_sql}
    UNION ALL
    {rule3_sql}
    UNION ALL
    {rule4_sql}
    """
    
    # Handle potential duplicates by selecting highest confidence matches
    final_sql = f"""
    WITH all_matches AS (
        {matches_sql}
    ),
    ranked_matches AS (
        SELECT 
            *,
            ROW_NUMBER() OVER (PARTITION BY ERP_TRANSACTION_ID ORDER BY MATCH_RULE, MATCH_CONFIDENCE DESC) as erp_rank,
            ROW_NUMBER() OVER (PARTITION BY CC_TRANSACTION_ID ORDER BY MATCH_RULE, MATCH_CONFIDENCE DESC) as cc_rank
        FROM all_matches
    )
    SELECT 
        *,
        CASE 
            WHEN MATCH_CONFIDENCE >= 0.95 THEN 'Auto-Match'
            WHEN MATCH_CONFIDENCE >= 0.8 THEN 'Suggested-Match'
            ELSE 'Potential-Match'
        END as MATCH_STATUS
    FROM ranked_matches
    WHERE erp_rank = 1 AND cc_rank = 1
    """
    
    # Execute and save results
    result_df = session.sql(final_sql)
    result_df.write.mode("overwrite").save_as_table(output_table)
    
    # Get match statistics
    stats_sql = f"""
    SELECT
        MATCH_STATUS,
        COUNT(*) as MATCH_COUNT,
        AVG(MATCH_CONFIDENCE) as AVG_CONFIDENCE,
        MIN(MATCH_CONFIDENCE) as MIN_CONFIDENCE,
        MAX(MATCH_CONFIDENCE) as MAX_CONFIDENCE
    FROM {output_table}
    GROUP BY MATCH_STATUS
    ORDER BY MATCH_STATUS
    """
    
    stats_df = session.sql(stats_sql)
    stats_rows = stats_df.collect()
    
    # Fix: Convert Row objects to dictionaries properly
    stats_dicts = []
    for row in stats_rows:
        # Create a dictionary with each column name and value
        row_dict = {}
        for col_name in stats_df.columns:
            row_dict[col_name] = row[col_name]
        stats_dicts.append(row_dict)
    
    # Count unmatched transactions
    unmatched_sql = f"""
    SELECT
        'Unmatched ERP Credit Card' as CATEGORY,
        COUNT(*) as COUNT
    FROM {erp_table} e
    WHERE e.PAYMENT_METHOD = 'Credit Card'
    AND NOT EXISTS (
        SELECT 1 FROM {output_table} m
        WHERE m.ERP_TRANSACTION_ID = e.TRANSACTION_ID
    )
    
    UNION ALL
    
    SELECT
        'Unmatched Credit Card' as CATEGORY,
        COUNT(*) as COUNT
    FROM {cc_table} c
    WHERE NOT EXISTS (
        SELECT 1 FROM {output_table} m
        WHERE m.CC_TRANSACTION_ID = c.TRANSACTION_ID
    )
    """
    
    unmatched_df = session.sql(unmatched_sql)
    unmatched_rows = unmatched_df.collect()
    
    # Fix: Convert Row objects to dictionaries properly
    unmatched_dicts = []
    for row in unmatched_rows:
        # Create a dictionary with each column name and value
        row_dict = {}
        for col_name in unmatched_df.columns:
            row_dict[col_name] = row[col_name]
        unmatched_dicts.append(row_dict)
    
    # Return summary
    match_results = {
        "match_statistics": stats_dicts,
        "unmatched_statistics": unmatched_dicts,
        "source_tables": [erp_table, cc_table],
        "target_table": output_table
    }
    
    return json.dumps(match_results)



# RECONCILIATION AND REPORTING
-----------------------------------------------------------------------------



In [10]:
#| export

def generate_reconciliation_report(session: Session,
                                  bank_match_table: str,
                                  cc_match_table: str,
                                  erp_table: str,
                                  bank_table: str,
                                  cc_table: str,
                                  output_table: str,
                                  as_of_date: str = None) -> str:
    """
    Generates a comprehensive reconciliation report.
    
    Parameters:
        session (Session): Snowflake session
        bank_match_table (str): Table with bank matching results
        cc_match_table (str): Table with credit card matching results
        erp_table (str): Standardized ERP transactions table
        bank_table (str): Standardized bank transactions table
        cc_table (str): Standardized credit card transactions table
        output_table (str): Output table for reconciliation report
        as_of_date (str): Optional date for point-in-time reconciliation (YYYY-MM-DD)
        
    Returns:
        str: Reconciliation summary
    """
    # Handle date filtering
    date_filter = ""
    if as_of_date:
        date_filter = f"AND e.TRANSACTION_DATE_STD <= '{as_of_date}'"
    
    # Create account summary SQL - restructured to avoid the WITH clause issue
    account_summary_sql = f"""
    SELECT
        ea.ACCOUNT_NUMBER,
        ea.ACCOUNT_NAME,
        ba.BANK_NAME,
        ea.ERP_TOTAL,
        ba.BANK_TOTAL,
        COALESCE(ea.ERP_TOTAL, 0) - COALESCE(ba.BANK_TOTAL, 0) as DIFFERENCE,
        COALESCE(ma.AUTO_MATCHED, 0) as AUTO_MATCHED_TOTAL,
        COALESCE(ma.SUGGESTED_MATCHED, 0) as SUGGESTED_MATCHED_TOTAL,
        COALESCE(ma.POTENTIAL_MATCHED, 0) as POTENTIAL_MATCHED_TOTAL,
        COALESCE(ue.UNMATCHED_ERP_COUNT, 0) as UNMATCHED_ERP_COUNT,
        COALESCE(ue.UNMATCHED_ERP_TOTAL, 0) as UNMATCHED_ERP_TOTAL,
        COALESCE(ub.UNMATCHED_BANK_COUNT, 0) as UNMATCHED_BANK_COUNT,
        COALESCE(ub.UNMATCHED_BANK_TOTAL, 0) as UNMATCHED_BANK_TOTAL,
        CASE 
            WHEN ABS(COALESCE(ea.ERP_TOTAL, 0) - COALESCE(ba.BANK_TOTAL, 0)) < 0.01 THEN 'Reconciled'
            WHEN ABS(COALESCE(ea.ERP_TOTAL, 0) - COALESCE(ba.BANK_TOTAL, 0)) <= 
                 ABS(COALESCE(ue.UNMATCHED_ERP_TOTAL, 0) - COALESCE(ub.UNMATCHED_BANK_TOTAL, 0)) THEN 'Explainable Difference'
            ELSE 'Unexplained Difference'
        END as RECONCILIATION_STATUS,
        CURRENT_TIMESTAMP() as REPORT_GENERATED_AT,
        '{as_of_date if as_of_date else "Current"}' as AS_OF_DATE
    FROM (
        -- ERP amounts by account
        SELECT
            e.ACCOUNT_NUMBER_STD as ACCOUNT_NUMBER,
            e.ACCOUNT_NAME as ACCOUNT_NAME,
            SUM(e.AMOUNT_STD) as ERP_TOTAL
        FROM {erp_table} e
        WHERE 1=1 {date_filter}
        GROUP BY e.ACCOUNT_NUMBER_STD, e.ACCOUNT_NAME
    ) ea
    FULL OUTER JOIN (
        -- Bank amounts by account
        SELECT
            b.ACCOUNT_NUMBER_STD as ACCOUNT_NUMBER,
            b.BANK_NAME as BANK_NAME,
            SUM(b.AMOUNT_STD) as BANK_TOTAL
        FROM {bank_table} b
        WHERE 1=1 {date_filter.replace('e.', 'b.')}
        GROUP BY b.ACCOUNT_NUMBER_STD, b.BANK_NAME
    ) ba ON ea.ACCOUNT_NUMBER = ba.ACCOUNT_NUMBER
    LEFT JOIN (
        -- Matched amounts by account
        SELECT
            e.ACCOUNT_NUMBER_STD as ACCOUNT_NUMBER,
            SUM(CASE WHEN m.MATCH_STATUS = 'Auto-Match' THEN e.AMOUNT_STD ELSE 0 END) as AUTO_MATCHED,
            SUM(CASE WHEN m.MATCH_STATUS = 'Suggested-Match' THEN e.AMOUNT_STD ELSE 0 END) as SUGGESTED_MATCHED,
            SUM(CASE WHEN m.MATCH_STATUS = 'Potential-Match' THEN e.AMOUNT_STD ELSE 0 END) as POTENTIAL_MATCHED
        FROM {bank_match_table} m
        JOIN {erp_table} e ON m.ERP_TRANSACTION_ID = e.TRANSACTION_ID
        WHERE 1=1 {date_filter}
        GROUP BY e.ACCOUNT_NUMBER_STD
    ) ma ON ea.ACCOUNT_NUMBER = ma.ACCOUNT_NUMBER
    LEFT JOIN (
        -- Unmatched ERP transactions by account
        SELECT
            e.ACCOUNT_NUMBER_STD as ACCOUNT_NUMBER,
            COUNT(*) as UNMATCHED_ERP_COUNT,
            SUM(e.AMOUNT_STD) as UNMATCHED_ERP_TOTAL
        FROM {erp_table} e
        WHERE NOT EXISTS (
            SELECT 1 FROM {bank_match_table} m
            WHERE m.ERP_TRANSACTION_ID = e.TRANSACTION_ID
        )
        AND e.PAYMENT_METHOD != 'Credit Card'
        {date_filter}
        GROUP BY e.ACCOUNT_NUMBER_STD
    ) ue ON ea.ACCOUNT_NUMBER = ue.ACCOUNT_NUMBER
    LEFT JOIN (
        -- Unmatched bank transactions by account
        SELECT
            b.ACCOUNT_NUMBER_STD as ACCOUNT_NUMBER,
            COUNT(*) as UNMATCHED_BANK_COUNT,
            SUM(b.AMOUNT_STD) as UNMATCHED_BANK_TOTAL
        FROM {bank_table} b
        WHERE NOT EXISTS (
            SELECT 1 FROM {bank_match_table} m
            WHERE m.BANK_TRANSACTION_ID = b.TRANSACTION_ID
        )
        {date_filter.replace('e.', 'b.')}
        GROUP BY b.ACCOUNT_NUMBER_STD
    ) ub ON ea.ACCOUNT_NUMBER = ub.ACCOUNT_NUMBER
    """
    
    # Create credit card reconciliation SQL - restructured to avoid the WITH clause issue
    cc_summary_sql = f"""
    SELECT
        'CREDIT CARD RECONCILIATION' as ACCOUNT_NUMBER,
        'All Credit Cards' as ACCOUNT_NAME,
        'Various Issuers' as BANK_NAME,
        ecca.ERP_TOTAL,
        cca.CC_TOTAL,
        COALESCE(ecca.ERP_TOTAL, 0) - COALESCE(cca.CC_TOTAL, 0) as DIFFERENCE,
        COALESCE(mcca.AUTO_MATCHED, 0) as AUTO_MATCHED_TOTAL,
        COALESCE(mcca.SUGGESTED_MATCHED, 0) as SUGGESTED_MATCHED_TOTAL,
        COALESCE(mcca.POTENTIAL_MATCHED, 0) as POTENTIAL_MATCHED_TOTAL,
        COALESCE(uecca.UNMATCHED_ERP_COUNT, 0) as UNMATCHED_ERP_COUNT,
        COALESCE(uecca.UNMATCHED_ERP_TOTAL, 0) as UNMATCHED_ERP_TOTAL,
        COALESCE(ucc.UNMATCHED_CC_COUNT, 0) as UNMATCHED_BANK_COUNT,
        COALESCE(ucc.UNMATCHED_CC_TOTAL, 0) as UNMATCHED_BANK_TOTAL,
        CASE 
            WHEN ABS(COALESCE(ecca.ERP_TOTAL, 0) - COALESCE(cca.CC_TOTAL, 0)) < 0.01 THEN 'Reconciled'
            WHEN ABS(COALESCE(ecca.ERP_TOTAL, 0) - COALESCE(cca.CC_TOTAL, 0)) <= 
                 ABS(COALESCE(uecca.UNMATCHED_ERP_TOTAL, 0) - COALESCE(ucc.UNMATCHED_CC_TOTAL, 0)) THEN 'Explainable Difference'
            ELSE 'Unexplained Difference'
        END as RECONCILIATION_STATUS,
        CURRENT_TIMESTAMP() as REPORT_GENERATED_AT,
        '{as_of_date if as_of_date else "Current"}' as AS_OF_DATE
    FROM (
        -- ERP credit card amounts
        SELECT
            'Credit Card' as CATEGORY,
            SUM(e.AMOUNT_STD) as ERP_TOTAL
        FROM {erp_table} e
        WHERE e.PAYMENT_METHOD = 'Credit Card'
        {date_filter}
    ) ecca
    FULL OUTER JOIN (
        -- Credit card statement amounts
        SELECT
            'Credit Card' as CATEGORY,
            SUM(c.AMOUNT_STD) as CC_TOTAL
        FROM {cc_table} c
        WHERE 1=1 {date_filter.replace('e.', 'c.')}
    ) cca ON ecca.CATEGORY = cca.CATEGORY
    LEFT JOIN (
        -- Matched credit card amounts
        SELECT
            'Credit Card' as CATEGORY,
            SUM(CASE WHEN m.MATCH_STATUS = 'Auto-Match' THEN e.AMOUNT_STD ELSE 0 END) as AUTO_MATCHED,
            SUM(CASE WHEN m.MATCH_STATUS = 'Suggested-Match' THEN e.AMOUNT_STD ELSE 0 END) as SUGGESTED_MATCHED,
            SUM(CASE WHEN m.MATCH_STATUS = 'Potential-Match' THEN e.AMOUNT_STD ELSE 0 END) as POTENTIAL_MATCHED
        FROM {cc_match_table} m
        JOIN {erp_table} e ON m.ERP_TRANSACTION_ID = e.TRANSACTION_ID
        WHERE 1=1 {date_filter}
    ) mcca ON ecca.CATEGORY = mcca.CATEGORY
    LEFT JOIN (
        -- Unmatched ERP credit card transactions
        SELECT
            'Credit Card' as CATEGORY,
            COUNT(*) as UNMATCHED_ERP_COUNT,
            SUM(e.AMOUNT_STD) as UNMATCHED_ERP_TOTAL
        FROM {erp_table} e
        WHERE e.PAYMENT_METHOD = 'Credit Card'
        AND NOT EXISTS (
            SELECT 1 FROM {cc_match_table} m
            WHERE m.ERP_TRANSACTION_ID = e.TRANSACTION_ID
        )
        {date_filter}
    ) uecca ON ecca.CATEGORY = uecca.CATEGORY
    LEFT JOIN (
        -- Unmatched credit card transactions
        SELECT
            'Credit Card' as CATEGORY,
            COUNT(*) as UNMATCHED_CC_COUNT,
            SUM(c.AMOUNT_STD) as UNMATCHED_CC_TOTAL
        FROM {cc_table} c
        WHERE NOT EXISTS (
            SELECT 1 FROM {cc_match_table} m
            WHERE m.CC_TRANSACTION_ID = c.TRANSACTION_ID
        )
        {date_filter.replace('e.', 'c.')}
    ) ucc ON ecca.CATEGORY = ucc.CATEGORY
    """
    
    # Combine bank and credit card reconciliation
    final_sql = f"""
    {account_summary_sql}
    UNION ALL
    {cc_summary_sql}
    """
    
    # Execute and save results
    result_df = session.sql(final_sql)
    result_df.write.mode("overwrite").save_as_table(output_table)
    
    # Get summary of reconciliation statuses
    summary_sql = f"""
    SELECT
        RECONCILIATION_STATUS,
        COUNT(*) as ACCOUNT_COUNT,
        SUM(ABS(DIFFERENCE)) as TOTAL_DIFFERENCE
    FROM {output_table}
    GROUP BY RECONCILIATION_STATUS
    ORDER BY RECONCILIATION_STATUS
    """
    
    summary_df = session.sql(summary_sql)
    summary_rows = summary_df.collect()
    
    # Fix: Convert Row objects to dictionaries properly
    summary_dicts = []
    for row in summary_rows:
        row_dict = {}
        for col_name in summary_df.columns:
            row_dict[col_name] = row[col_name]
        summary_dicts.append(row_dict)
    
    # Return reconciliation summary
    recon_summary = {
        "reconciliation_summary": summary_dicts,
        "as_of_date": as_of_date if as_of_date else "Current",
        "source_tables": [bank_match_table, cc_match_table, erp_table, bank_table, cc_table],
        "output_table": output_table,
        "report_generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    
    return json.dumps(recon_summary)

In [24]:
#| export

def identify_reconciliation_exceptions(session: Session,
                                      recon_report_table: str,
                                      bank_match_table: str,
                                      cc_match_table: str,
                                      output_table: str) -> str:
    """
    Identifies and categorizes reconciliation exceptions requiring investigation.
    
    Parameters:
        session (Session): Snowflake session
        recon_report_table (str): Reconciliation report table
        bank_match_table (str): Bank match results table
        cc_match_table (str): Credit card match results table
        output_table (str): Output table for exceptions
        
    Returns:
        str: Exception summary
    """
    # Create SQL to identify exceptions
    # For simplicity, we'll just use the reconciliation report to identify exceptions
    exceptions_sql = f"""
    SELECT
        uuid_string() as EXCEPTION_ID,
        ACCOUNT_NUMBER,
        ACCOUNT_NAME,
        BANK_NAME,
        CASE
            WHEN RECONCILIATION_STATUS = 'Unexplained Difference' THEN 'Unexplained Difference'
            WHEN UNMATCHED_ERP_COUNT > 0 AND UNMATCHED_ERP_TOTAL > 1000 THEN 'Large Unmatched ERP Transactions'
            WHEN UNMATCHED_BANK_COUNT > 0 AND UNMATCHED_BANK_TOTAL > 1000 THEN 'Large Unmatched Bank Transactions'
            ELSE 'Other Discrepancy'
        END as EXCEPTION_TYPE,
        CASE
            WHEN ABS(DIFFERENCE) > 50000 THEN 'HIGH'
            WHEN ABS(DIFFERENCE) > 10000 THEN 'MEDIUM'
            ELSE 'LOW'
        END as PRIORITY,
        CASE
            WHEN RECONCILIATION_STATUS = 'Unexplained Difference' THEN
                CASE 
                    WHEN ABS(DIFFERENCE) > 10000 THEN 'Major Discrepancy'
                    WHEN ABS(DIFFERENCE) > 1000 THEN 'Significant Discrepancy'
                    ELSE 'Minor Discrepancy'
                END
            WHEN UNMATCHED_ERP_COUNT > 0 THEN 'Unmatched Transactions'
            WHEN UNMATCHED_BANK_COUNT > 0 THEN 'Unmatched Transactions'
            ELSE 'Other Issue'
        END as EXCEPTION_CATEGORY,
        CASE
            WHEN RECONCILIATION_STATUS = 'Unexplained Difference' THEN 'Investigate balance discrepancy that cannot be explained by unmatched transactions'
            WHEN UNMATCHED_ERP_COUNT > 0 THEN 'Investigate unmatched ERP transactions'
            WHEN UNMATCHED_BANK_COUNT > 0 THEN 'Investigate unmatched bank transactions'
            ELSE 'Review reconciliation data'
        END as RECOMMENDED_ACTION,
        'Open' as STATUS,
        NULL as ASSIGNED_TO,
        NULL as RESOLUTION,
        NULL as RESOLUTION_DATE,
        CURRENT_TIMESTAMP() as IDENTIFIED_AT
    FROM {recon_report_table}
    WHERE RECONCILIATION_STATUS <> 'Reconciled'
       OR UNMATCHED_ERP_COUNT > 0
       OR UNMATCHED_BANK_COUNT > 0
    """
    
    # Execute and save results
    exceptions_df = session.sql(exceptions_sql)
    exceptions_df.write.mode("overwrite").save_as_table(output_table)
    
    # Get exception summary
    summary_sql = f"""
    SELECT
        EXCEPTION_TYPE,
        PRIORITY,
        COUNT(*) as EXCEPTION_COUNT
    FROM {output_table}
    GROUP BY EXCEPTION_TYPE, PRIORITY
    ORDER BY PRIORITY, EXCEPTION_TYPE
    """
    
    summary_df = session.sql(summary_sql)
    summary_rows = summary_df.collect()
    
    # Fix: Convert Row objects to dictionaries properly
    summary_dicts = []
    for row in summary_rows:
        row_dict = {}
        for col_name in summary_df.columns:
            row_dict[col_name] = row[col_name]
        summary_dicts.append(row_dict)
    
    # Return exception summary
    exception_summary = {
        "exception_summary": summary_dicts,
        "total_exceptions": session.sql(f"SELECT COUNT(*) as TOTAL FROM {output_table}").collect()[0]["TOTAL"],
        "source_tables": [recon_report_table, bank_match_table, cc_match_table],
        "output_table": output_table,
        "identified_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    
    return json.dumps(exception_summary)

In [38]:
#| export

def run_financial_matching_pipeline(session: Session, 
                                   config: dict) -> str:
    """
    Orchestrates the entire financial transaction matching and reconciliation pipeline.
    
    Parameters:
        session (Session): Snowflake session
        config (dict): Configuration dictionary with parameters for the pipeline:
            - num_transactions: Number of test transactions to generate
            - error_rate: Rate of errors to introduce
            - date_range_start: Start date for transactions
            - date_range_end: End date for transactions
            - as_of_date: Date for reconciliation point-in-time (optional)
            - table_prefix: Prefix for all generated tables
            - run_id: Unique identifier for the pipeline run
            
    Returns:
        str: Summary of the pipeline execution
    """
    if isinstance(config, str):
        try:
            config = json.loads(config)
        except json.JSONDecodeError:
            raise ValueError("Invalid JSON string provided for config")
    
    if not isinstance(config, dict):
        raise TypeError("Config must be a dictionary or a JSON string")

    # Extract configuration parameters
    num_transactions = config.get("num_transactions", 10000)
    error_rate = config.get("error_rate", 0.1)
    date_range_start = config.get("date_range_start", "2023-01-01")
    date_range_end = config.get("date_range_end", "2023-12-31")
    as_of_date = config.get("as_of_date")
    table_prefix = config.get("table_prefix", "FINMATCH")
    run_id = config.get("run_id", datetime.now().strftime("%Y%m%d%H%M%S"))
    
    # Define table names
    tables = {
        "erp_raw": f"{table_prefix}_ERP_RAW_{run_id}",
        "bank_raw": f"{table_prefix}_BANK_RAW_{run_id}",
        "cc_raw": f"{table_prefix}_CC_RAW_{run_id}",
        "erp_std": f"{table_prefix}_ERP_STD_{run_id}",
        "bank_std": f"{table_prefix}_BANK_STD_{run_id}",
        "cc_std": f"{table_prefix}_CC_STD_{run_id}",
        "bank_matches": f"{table_prefix}_BANK_MATCHES_{run_id}",
        "cc_matches": f"{table_prefix}_CC_MATCHES_{run_id}",
        "recon_report": f"{table_prefix}_RECON_REPORT_{run_id}",
        "exceptions": f"{table_prefix}_EXCEPTIONS_{run_id}"
    }
    
    # Initialize results dictionary
    results = {
        "run_id": run_id,
        "pipeline_start": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "configuration": config,
        "tables_created": tables,
        "steps": []
    }
    
    try:
        # Step 1: Generate test data
        step_start = datetime.now()
        generate_result = generate_financial_data(
            session=session,
            num_transactions=num_transactions,
            error_rate=error_rate,
            date_range_start=date_range_start,
            date_range_end=date_range_end,
            erp_table=tables["erp_raw"],
            bank_table=tables["bank_raw"],
            credit_card_table=tables["cc_raw"]
        )
        step_result = {
            "step": "Data Generation",
            "status": "Success",
            "duration_seconds": (datetime.now() - step_start).total_seconds(),
            "details": json.loads(generate_result)
        }
        results["steps"].append(step_result)
        
        # Step 2: Standardize data
        step_start = datetime.now()
        standardize_erp_transactions(session, tables["erp_raw"], tables["erp_std"])
        standardize_bank_transactions(session, tables["bank_raw"], tables["bank_std"])
        standardize_cc_transactions(session, tables["cc_raw"], tables["cc_std"])
        step_result = {
            "step": "Data Standardization",
            "status": "Success",
            "duration_seconds": (datetime.now() - step_start).total_seconds(),
            "details": {
                "tables_standardized": [tables["erp_std"], tables["bank_std"], tables["cc_std"]]
            }
        }
        results["steps"].append(step_result)
        
        # Step 3: Match transactions
        step_start = datetime.now()
        bank_match_result = match_erp_to_bank(
            session=session,
            erp_table=tables["erp_std"],
            bank_table=tables["bank_std"],
            output_table=tables["bank_matches"],
            date_variance=3
        )
        cc_match_result = match_erp_to_credit_card(
            session=session,
            erp_table=tables["erp_std"],
            cc_table=tables["cc_std"],
            output_table=tables["cc_matches"],
            date_variance=5
        )
        step_result = {
            "step": "Transaction Matching",
            "status": "Success",
            "duration_seconds": (datetime.now() - step_start).total_seconds(),
            "details": {
                "bank_matching": json.loads(bank_match_result),
                "credit_card_matching": json.loads(cc_match_result)
            }
        }
        results["steps"].append(step_result)
        
        # Step 4: Generate reconciliation report
        step_start = datetime.now()
        recon_result = generate_reconciliation_report(
            session=session,
            bank_match_table=tables["bank_matches"],
            cc_match_table=tables["cc_matches"],
            erp_table=tables["erp_std"],
            bank_table=tables["bank_std"],
            cc_table=tables["cc_std"],
            output_table=tables["recon_report"],
            as_of_date=as_of_date
        )
        step_result = {
            "step": "Reconciliation Report Generation",
            "status": "Success",
            "duration_seconds": (datetime.now() - step_start).total_seconds(),
            "details": json.loads(recon_result)
        }
        results["steps"].append(step_result)
        
        # Step 5: Identify exceptions
        step_start = datetime.now()
        exceptions_result = identify_reconciliation_exceptions(
            session=session,
            recon_report_table=tables["recon_report"],
            bank_match_table=tables["bank_matches"],
            cc_match_table=tables["cc_matches"],
            output_table=tables["exceptions"]
        )
        step_result = {
            "step": "Exception Identification",
            "status": "Success",
            "duration_seconds": (datetime.now() - step_start).total_seconds(),
            "details": json.loads(exceptions_result)
        }
        results["steps"].append(step_result)
        
        # Record pipeline completion
        results["pipeline_end"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        results["pipeline_duration_seconds"] = (datetime.now() - datetime.strptime(results["pipeline_start"], "%Y-%m-%d %H:%M:%S")).total_seconds()
        results["pipeline_status"] = "Success"
        
    except Exception as e:
        # Record failure
        results["pipeline_end"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        results["pipeline_duration_seconds"] = (datetime.now() - datetime.strptime(results["pipeline_start"], "%Y-%m-%d %H:%M:%S")).total_seconds()
        results["pipeline_status"] = "Failed"
        results["error"] = str(e)
    
    return json.dumps(results)

# Test Notebook

In [ ]:
#| skip
from customfunctions.connection import SnowflakeConnection
from snowflake.snowpark.version import VERSION

import os


# Establish a connection to Snowflake


In [ ]:
#| skip
try:
    # Try to get active session
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception as e:
    print(f"No active session, creating a new one. Error: {e}")
    from customfunctions.connection import SnowflakeConnection
    
    # Create connection using environment variables or hard-coded for testing
    config = {
        'user': os.getenv('SNOWFLAKE_USER', ''),
        'password': os.getenv('SNOWFLAKE_PASSWORD', ''),
        'account': os.getenv('SNOWFLAKE_ACCOUNT', ''),
        'database': 'DATASCIENCE',
        'warehouse': 'DS_WH_XS',
        'schema': 'CUSTOM_FUNCTIONS',
        'role': 'DATA_SCIENTIST'
    }
    
    sfc = SnowflakeConnection(**config)
    session = sfc.get_session()
    print("Connected to Snowflake")

snowflake_environment = session.sql('SELECT current_user(), current_version()').collect()
snowpark_version = VERSION
print('\nConnection Established with the following parameters:')
print('Role                        : {}'.format(session.get_current_role()))
print('Database                    : {}'.format(session.get_current_database()))
print('Schema                      : {}'.format(session.get_current_schema()))
print('Warehouse                   : {}'.format(session.get_current_warehouse()))
print('Snowflake version           : {}'.format(snowflake_environment[0][1]))
print('Snowpark for Python version : {}.{}.{}'.format(snowpark_version[0],snowpark_version[1],snowpark_version[2]))


No active session, creating a new one. Error: (1403): No default Session is found. Please create a session before you call function 'udf' or use decorator '@udf'.
Connected to Snowflake

Connection Established with the following parameters:
Role                        : "DATA_SCIENTIST"
Database                    : "DATASCIENCE"
Schema                      : "CUSTOM_FUNCTIONS"
Warehouse                   : "DS_WH_XS"
Snowflake version           : 9.11.2
Snowpark for Python version : 1.26.0



# Define test table names with a unique prefix


In [ ]:
#| skip
run_id = "NOTEBOOK_TEST_001"
tables = {
    "erp_raw": f"TEST_ERP_RAW_{run_id}",
    "bank_raw": f"TEST_BANK_RAW_{run_id}",
    "cc_raw": f"TEST_CC_RAW_{run_id}",
    "erp_std": f"TEST_ERP_STD_{run_id}",
    "bank_std": f"TEST_BANK_STD_{run_id}",
    "cc_std": f"TEST_CC_STD_{run_id}",
    "bank_matches": f"TEST_BANK_MATCHES_{run_id}",
    "cc_matches": f"TEST_CC_MATCHES_{run_id}",
    "recon_report": f"TEST_RECON_REPORT_{run_id}",
    "exceptions": f"TEST_EXCEPTIONS_{run_id}"
}



# Step 1: Generate test data (a smaller dataset for faster testing)


In [ ]:
#| skip
print("Generating test data...")
result = generate_financial_data(
    session=session,
    num_transactions=1000,  # Smaller number for testing
    error_rate=0.2,
    date_range_start="2023-01-01",
    date_range_end="2023-12-31",
    erp_table=tables["erp_raw"],
    bank_table=tables["bank_raw"],
    credit_card_table=tables["cc_raw"]
)
print(f"Data generation result: {result}")

Generating test data...
Data generation result: {"total_base_transactions": 1000, "erp_transactions": 1000, "bank_transactions": 619, "credit_card_transactions": 163, "error_rate_applied": 0.2, "date_range": "2023-01-01 to 2023-12-31", "tables_created": ["TEST_ERP_RAW_NOTEBOOK_TEST_001", "TEST_BANK_RAW_NOTEBOOK_TEST_001", "TEST_CC_RAW_NOTEBOOK_TEST_001"]}



# Step 2: Standardize data


In [ ]:
#| skip
print("\nStandardizing data...")
erp_result = standardize_erp_transactions(session, tables["erp_raw"], tables["erp_std"])
bank_result = standardize_bank_transactions(session, tables["bank_raw"], tables["bank_std"])
cc_result = standardize_cc_transactions(session, tables["cc_raw"], tables["cc_std"])
print(f"ERP standardization: {erp_result}")
print(f"Bank standardization: {bank_result}")
print(f"Credit card standardization: {cc_result}")



Standardizing data...
ERP standardization: Standardized ERP transactions from TEST_ERP_RAW_NOTEBOOK_TEST_001 and saved to TEST_ERP_STD_NOTEBOOK_TEST_001
Bank standardization: Standardized bank transactions from TEST_BANK_RAW_NOTEBOOK_TEST_001 and saved to TEST_BANK_STD_NOTEBOOK_TEST_001
Credit card standardization: Standardized credit card transactions from TEST_CC_RAW_NOTEBOOK_TEST_001 and saved to TEST_CC_STD_NOTEBOOK_TEST_001



# Step 3: Match transactions


In [ ]:
#| skip
print("\nMatching transactions...")
bank_match_result = match_erp_to_bank(
    session=session,
    erp_table=tables["erp_std"],
    bank_table=tables["bank_std"],
    output_table=tables["bank_matches"],
    date_variance=3
)
print(f"Bank matching result: {bank_match_result}")



Matching transactions...
Bank matching result: {"match_statistics": [{"MATCH_STATUS": "Auto-Match", "MATCH_COUNT": 483, "AVG_CONFIDENCE": 0.9901656314699793, "MIN_CONFIDENCE": 0.95, "MAX_CONFIDENCE": 1.0}], "unmatched_statistics": [{"CATEGORY": "Unmatched ERP", "COUNT": 517}, {"CATEGORY": "Unmatched Bank", "COUNT": 136}], "source_tables": ["TEST_ERP_STD_NOTEBOOK_TEST_001", "TEST_BANK_STD_NOTEBOOK_TEST_001"], "target_table": "TEST_BANK_MATCHES_NOTEBOOK_TEST_001"}


In [ ]:
#| skip
cc_match_result = match_erp_to_credit_card(
    session=session,
    erp_table=tables["erp_std"],
    cc_table=tables["cc_std"],
    output_table=tables["cc_matches"],
    date_variance=5
)
print(f"Credit card matching result: {cc_match_result}")


Credit card matching result: {"match_statistics": [{"MATCH_STATUS": "Auto-Match", "MATCH_COUNT": 116, "AVG_CONFIDENCE": 0.9987068965517242, "MIN_CONFIDENCE": 0.95, "MAX_CONFIDENCE": 1.0}, {"MATCH_STATUS": "Potential-Match", "MATCH_COUNT": 26, "AVG_CONFIDENCE": 0.6629935162717076, "MIN_CONFIDENCE": 0.5775294117647058, "MAX_CONFIDENCE": 0.6956180171097814}, {"MATCH_STATUS": "Suggested-Match", "MATCH_COUNT": 21, "AVG_CONFIDENCE": 0.8999999999999999, "MIN_CONFIDENCE": 0.9, "MAX_CONFIDENCE": 0.9}], "unmatched_statistics": [{"CATEGORY": "Unmatched ERP Credit Card", "COUNT": 11}, {"CATEGORY": "Unmatched Credit Card", "COUNT": 0}], "source_tables": ["TEST_ERP_STD_NOTEBOOK_TEST_001", "TEST_CC_STD_NOTEBOOK_TEST_001"], "target_table": "TEST_CC_MATCHES_NOTEBOOK_TEST_001"}



# Step 4: Generate reconciliation report


In [ ]:
#| skip
print("\nGenerating reconciliation report...")
recon_result = generate_reconciliation_report(
    session=session,
    bank_match_table=tables["bank_matches"],
    cc_match_table=tables["cc_matches"],
    erp_table=tables["erp_std"],
    bank_table=tables["bank_std"],
    cc_table=tables["cc_std"],
    output_table=tables["recon_report"]
)
print(f"Reconciliation report result: {recon_result}")



Generating reconciliation report...
Reconciliation report result: {"reconciliation_summary": [{"RECONCILIATION_STATUS": "Explainable Difference", "ACCOUNT_COUNT": 2, "TOTAL_DIFFERENCE": 4493.02}, {"RECONCILIATION_STATUS": "Unexplained Difference", "ACCOUNT_COUNT": 4, "TOTAL_DIFFERENCE": 38847.28}], "as_of_date": "Current", "source_tables": ["TEST_BANK_MATCHES_NOTEBOOK_TEST_001", "TEST_CC_MATCHES_NOTEBOOK_TEST_001", "TEST_ERP_STD_NOTEBOOK_TEST_001", "TEST_BANK_STD_NOTEBOOK_TEST_001", "TEST_CC_STD_NOTEBOOK_TEST_001"], "output_table": "TEST_RECON_REPORT_NOTEBOOK_TEST_001", "report_generated_at": "2025-05-01 15:35:33"}



# Step 5: Identify exceptions


In [ ]:
#| skip
print("\nIdentifying exceptions...")
exceptions_result = identify_reconciliation_exceptions(
    session=session,
    recon_report_table=tables["recon_report"],
    bank_match_table=tables["bank_matches"],
    cc_match_table=tables["cc_matches"],
    output_table=tables["exceptions"]
)
print(f"Exceptions identification result: {exceptions_result}")



Identifying exceptions...
Exceptions identification result: {"exception_summary": [{"EXCEPTION_TYPE": "Large Unmatched Bank Transactions", "PRIORITY": "LOW", "EXCEPTION_COUNT": 1}, {"EXCEPTION_TYPE": "Large Unmatched ERP Transactions", "PRIORITY": "LOW", "EXCEPTION_COUNT": 1}, {"EXCEPTION_TYPE": "Unexplained Difference", "PRIORITY": "LOW", "EXCEPTION_COUNT": 2}, {"EXCEPTION_TYPE": "Unexplained Difference", "PRIORITY": "MEDIUM", "EXCEPTION_COUNT": 2}], "total_exceptions": 6, "source_tables": ["TEST_RECON_REPORT_NOTEBOOK_TEST_001", "TEST_BANK_MATCHES_NOTEBOOK_TEST_001", "TEST_CC_MATCHES_NOTEBOOK_TEST_001"], "output_table": "TEST_EXCEPTIONS_NOTEBOOK_TEST_001", "identified_at": "2025-05-01 15:39:34"}



# View sample data from generated tables


In [ ]:
#| skip
print("\nSample data from reconciliation report:")
session.sql(f"SELECT * FROM {tables['recon_report']} LIMIT 5").show()



Sample data from reconciliation report:
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"ACCOUNT_NUMBER"            |"ACCOUNT_NAME"     |"BANK_NAME"          |"ERP_TOTAL"          |"BANK_TOTAL"         |"DIFFERENCE"         |"AUTO_MATCHED_TOTAL"  |"SUGGESTED_MATCHED_TOTAL"  |"POTENTIAL_MATCHED_TOTAL"  |"UNMATCHED_ERP_COUNT"  |"UNMATCHED_ERP_TOTAL"  |"UNMATCHED_BANK_COUNT"  |"UNMATCHED_BANK_TOTAL"  |"RECONCILIATION_STATUS"  |"REPORT_GENERATED_AT"             |"AS_OF_DATE"  |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
#| skip
print("\nSample data from exceptions table:")
session.sql(f"SELECT * FROM {tables['exceptions']} LIMIT 5").show()



Sample data from exceptions table:
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"EXCEPTION_ID"                        |"ACCOUNT_NUMBER"            |"ACCOUNT_NAME"      |"BANK_NAME"          |"EXCEPTION_TYPE"                   |"PRIORITY"  |"EXCEPTION_CATEGORY"     |"RECOMMENDED_ACTION"                                |"STATUS"  |"ASSIGNED_TO"  |"RESOLUTION"  |"RESOLUTION_DATE"  |"IDENTIFIED_AT"                   |
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


# Check match statistics


In [ ]:
#| skip
print("\nMatch statistics summary:")
bank_match_stats = json.loads(bank_match_result)["match_statistics"]
print(f"Bank matches: {bank_match_stats}")



Match statistics summary:
Bank matches: [{'MATCH_STATUS': 'Auto-Match', 'MATCH_COUNT': 483, 'AVG_CONFIDENCE': 0.9901656314699793, 'MIN_CONFIDENCE': 0.95, 'MAX_CONFIDENCE': 1.0}]


In [ ]:
#| skip
cc_match_stats = json.loads(cc_match_result)["match_statistics"]
print(f"Credit card matches: {cc_match_stats}")


Credit card matches: [{'MATCH_STATUS': 'Auto-Match', 'MATCH_COUNT': 116, 'AVG_CONFIDENCE': 0.9987068965517242, 'MIN_CONFIDENCE': 0.95, 'MAX_CONFIDENCE': 1.0}, {'MATCH_STATUS': 'Potential-Match', 'MATCH_COUNT': 26, 'AVG_CONFIDENCE': 0.6629935162717076, 'MIN_CONFIDENCE': 0.5775294117647058, 'MAX_CONFIDENCE': 0.6956180171097814}, {'MATCH_STATUS': 'Suggested-Match', 'MATCH_COUNT': 21, 'AVG_CONFIDENCE': 0.8999999999999999, 'MIN_CONFIDENCE': 0.9, 'MAX_CONFIDENCE': 0.9}]



# Optional: Clean up test tables when done


In [ ]:
#| skip
# # Uncomment these lines if you want to automatically clean up
# for table in tables.values():
#     session.sql(f"DROP TABLE IF EXISTS {table}").collect()
# print("Test tables cleaned up")

# Call the ``run_financial_matching_pipeline`` directly


In [39]:
#| skip
pipeline_config = {
    "num_transactions": 5000,  # Smaller for testing
    "error_rate": 0.1,
    "date_range_start": "2023-01-01",
    "date_range_end": "2023-12-31",
    "table_prefix": "NOTEBOOK",
    "run_id": "DIRECT_CALL_002"
}

result = run_financial_matching_pipeline(session, pipeline_config)
result_json = json.loads(result)

# Analyze results
print(f"Pipeline status: {result_json['pipeline_status']}")
print(f"Duration: {result_json['pipeline_duration_seconds']} seconds")

# Print results from each step
for step in result_json['steps']:
    print(f"Step: {step['step']}")
    print(f"Status: {step['status']}")
    print(f"Duration: {step['duration_seconds']} seconds")
    print("---")


Pipeline status: Success
Duration: 31.016229 seconds
Step: Data Generation
Status: Success
Duration: 13.558642 seconds
---
Step: Data Standardization
Status: Success
Duration: 4.044402 seconds
---
Step: Transaction Matching
Status: Success
Duration: 8.014365 seconds
---
Step: Reconciliation Report Generation
Status: Success
Duration: 2.278069 seconds
---
Step: Exception Identification
Status: Success
Duration: 2.685987 seconds
---


In [40]:
#| hide
import nbdev; nbdev.nbdev_export()